## Transformação de dados - compras_ordens

#### 1.Carregar tabela do bronze

In [ ]:
import sys
sys.path.append("/app")


from utils import create_spark_session, load_config, save_table
from pyspark.sql import functions as F
from delta.tables import DeltaTable

tabela_nome = "compras_ordens"

spark = create_spark_session("compras_ordens")
config = load_config()

# ler bronze
df = spark.read.format("delta").load(
    f"data/bronze/{tabela_nome}"
)

#### 2.Executar transformação

In [ ]:

# transformação
df_silver = (
    df
    .select("id", "status", "empresa_id", "fornecedor_id", "funcionario_id")
)

#### 3.Armazenar dados na camada silver

In [ ]:
# salvar silver
path = f"data/silver/{tabela_nome}"

if DeltaTable.isDeltaTable(spark, path):

    delta_table = DeltaTable.forPath(spark, path)

    (
        delta_table.alias("t")
        .merge(
            df_silver.alias("s"),
            """
            t.id = s.id
            """
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )

else:

    df_silver.write.format("delta").save(path)

#Manter arquivos antigos por 7 dias (168 horas)
spark.sql(f"VACUUM delta.`{path}` RETAIN 168 HOURS")